In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
import joblib
import os
import threading
import time

# Для Flask и Prometheus
from flask import Flask, request, jsonify
from prometheus_client import Counter, Gauge, generate_latest, REGISTRY
from prometheus_flask_exporter import PrometheusMetrics



df = pd.read_csv(r'C:\Users\Kohov\OneDrive\Desktop\IDE\Ds_prod_proj\data\winequality-red (1).csv', sep=';')
print(df.head())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2      9.8        5 

In [25]:
# Преобразуем качество в бинарный класс: 1 если >=6, иначе 0
df['good'] = (df['quality'] >= 6).astype(int)

# Признаки и цель
X = df.drop(['quality', 'good'], axis=1)
y = df['good']

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделение на обучающую и тестовую (для оценки)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Сохраним scaler для использования в сервисе
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [26]:
# Обучаем случайный лес
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

# Сохраняем модель
joblib.dump(model, 'model.pkl')

# Оцениваем на тестовой выборке (эти метрики будем экспортировать)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f'Initial metrics - Accuracy: {accuracy:.3f}, F1: {f1:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}')

Initial metrics - Accuracy: 0.791, F1: 0.813, Precision: 0.811, Recall: 0.816


In [ ]:
import threading
import time
import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from prometheus_flask_exporter import PrometheusMetrics
from prometheus_client import Gauge, Counter, REGISTRY
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def safe_gauge(name, doc, labelnames=()):
    if name in REGISTRY._names_to_collectors:
        metric = REGISTRY._names_to_collectors[name]
        if isinstance(metric, Gauge):
            return metric
        else:
            REGISTRY.unregister(metric)
    return Gauge(name, doc, labelnames=labelnames)

def safe_counter(name, doc, labelnames=()):
    if name in REGISTRY._names_to_collectors:
        metric = REGISTRY._names_to_collectors[name]
        if isinstance(metric, Counter):
            return metric
        else:
            REGISTRY.unregister(metric)
    return Counter(name, doc, labelnames=labelnames)

app = Flask(__name__)
metrics = PrometheusMetrics(app)

accuracy_gauge = safe_gauge('model_accuracy', 'Accuracy of the model on test set')
f1_gauge = safe_gauge('model_f1', 'F1 score on test set')
precision_gauge = safe_gauge('model_precision', 'Precision on test set')
recall_gauge = safe_gauge('model_recall', 'Recall on test set')
prediction_counter = safe_counter('model_predictions_total', 'Total number of predictions')


current_model = None
current_scaler = None
X_train_all = None
y_train_all = None
X_test_global = None
y_test_global = None


def load_model_and_scaler():
    global current_model, current_scaler
    current_model = joblib.load('model.pkl')
    current_scaler = joblib.load('scaler.pkl')

def load_training_data():
    global X_train_all, y_train_all, X_test_global, y_test_global
    # Предполагается, что переменные X_train, y_train, X_test, y_test уже существуют
    X_train_all = X_train
    y_train_all = y_train
    X_test_global = X_test
    y_test_global = y_test

def update_metrics():
    global current_model, X_test_global, y_test_global
    if current_model is None or X_test_global is None:
        return
    y_pred = current_model.predict(X_test_global)
    acc = accuracy_score(y_test_global, y_pred)
    f1 = f1_score(y_test_global, y_pred)
    prec = precision_score(y_test_global, y_pred)
    rec = recall_score(y_test_global, y_pred)
    accuracy_gauge.set(acc)
    f1_gauge.set(f1)
    precision_gauge.set(prec)
    recall_gauge.set(rec)

# Инициализация
load_model_and_scaler()
load_training_data()
update_metrics()

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features = np.array(data['features']).reshape(1, -1)
    features_scaled = current_scaler.transform(features)
    pred = current_model.predict(features_scaled)[0]
    proba = current_model.predict_proba(features_scaled)[0].tolist()
    prediction_counter.inc()
    return jsonify({'prediction': int(pred), 'probability': proba})

@app.route('/retrain', methods=['POST'])
def retrain():
    global current_model, X_train_all, y_train_all
    data = request.get_json()
    X_new = np.array(data['X_new'])
    y_new = np.array(data['y_new'])
    X_new_scaled = current_scaler.transform(X_new)
    # Добавляем новые данные к обучающей выборке
    X_train_all = np.vstack([X_train_all, X_new_scaled])
    y_train_all = np.hstack([y_train_all, y_new])
    # Переобучаем модель
    new_model = RandomForestClassifier(n_estimators=100, random_state=42)
    new_model.fit(X_train_all, y_train_all)
    joblib.dump(new_model, 'model.pkl')
    current_model = new_model
    update_metrics()
    return jsonify({'status': 'retrained successfully', 'new_train_size': len(X_train_all)})

@app.route('/metrics', methods=['GET'])
def metrics_endpoint():
    from prometheus_client import generate_latest
    return generate_latest()


def periodic_metric_update():
    while True:
        time.sleep(30)
        update_metrics()

if not app.testing:
    thread = threading.Thread(target=periodic_metric_update, daemon=True)
    thread.start()


from werkzeug.serving import make_server
server = make_server('0.0.0.0', 5000, app)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(" Flask сервер запущен на http://0.0.0.0:5000")
print(" Метрики доступны по адресу http://0.0.0.0:5000/metrics")

 Flask сервер запущен на http://0.0.0.0:5000
 Метрики доступны по адресу http://0.0.0.0:5000/metrics


In [28]:
import threading
import time
from flask import Flask, request, jsonify
from prometheus_flask_exporter import PrometheusMetrics
from prometheus_client import Gauge, Counter, REGISTRY

# Функции для создания метрик
def get_or_create_gauge(name, documentation, labelnames=()):
    if name in REGISTRY._names_to_collectors:
        metric = REGISTRY._names_to_collectors[name]
        if isinstance(metric, Gauge):
            return metric
        else:
            REGISTRY.unregister(metric)
    return Gauge(name, documentation, labelnames=labelnames)

def get_or_create_counter(name, documentation, labelnames=()):
    if name in REGISTRY._names_to_collectors:
        metric = REGISTRY._names_to_collectors[name]
        if isinstance(metric, Counter):
            return metric
        else:
            REGISTRY.unregister(metric)
    return Counter(name, documentation, labelnames=labelnames)

app = Flask(__name__)
metrics = PrometheusMetrics(app)

# Кастомные метрики
accuracy_gauge = get_or_create_gauge('model_accuracy', 'Accuracy of the model on test set')
f1_gauge = get_or_create_gauge('model_f1', 'F1 score on test set')
precision_gauge = get_or_create_gauge('model_precision', 'Precision on test set')
recall_gauge = get_or_create_gauge('model_recall', 'Recall on test set')
prediction_counter = get_or_create_counter('model_predictions_total', 'Total number of predictions')

# Глобальные переменные
current_model = None
current_scaler = None
X_train_all = None
y_train_all = None
X_test_global = None
y_test_global = None

def load_model_and_scaler():
    global current_model, current_scaler
    current_model = joblib.load('model.pkl')
    current_scaler = joblib.load('scaler.pkl')

def load_training_data():
    global X_train_all, y_train_all, X_test_global, y_test_global
    X_train_all = X_train  # из предыдущих ячеек
    y_train_all = y_train
    X_test_global = X_test
    y_test_global = y_test

def update_metrics():
    global current_model, X_test_global, y_test_global
    if current_model is None:
        return
    y_pred = current_model.predict(X_test_global)
    acc = accuracy_score(y_test_global, y_pred)
    f1 = f1_score(y_test_global, y_pred)
    prec = precision_score(y_test_global, y_pred)
    rec = recall_score(y_test_global, y_pred)
    accuracy_gauge.set(acc)
    f1_gauge.set(f1)
    precision_gauge.set(prec)
    recall_gauge.set(rec)

load_model_and_scaler()
load_training_data()
update_metrics()

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features = np.array(data['features']).reshape(1, -1)
    features_scaled = current_scaler.transform(features)
    pred = current_model.predict(features_scaled)[0]
    proba = current_model.predict_proba(features_scaled)[0].tolist()
    prediction_counter.inc()
    return jsonify({'prediction': int(pred), 'probability': proba})

@app.route('/retrain', methods=['POST'])
def retrain():
    global current_model, X_train_all, y_train_all, X_test_global, y_test_global
    data = request.get_json()
    X_new = np.array(data['X_new'])
    y_new = np.array(data['y_new'])
    X_new_scaled = current_scaler.transform(X_new)
    X_train_all = np.vstack([X_train_all, X_new_scaled])
    y_train_all = np.hstack([y_train_all, y_new])
    new_model = RandomForestClassifier(n_estimators=100, random_state=42)
    new_model.fit(X_train_all, y_train_all)
    joblib.dump(new_model, 'model.pkl')
    current_model = new_model
    update_metrics()
    return jsonify({'status': 'retrained successfully', 'new_train_size': len(X_train_all)})

def periodic_metric_update():
    while True:
        time.sleep(30)
        update_metrics()

if not app.testing:
    thread = threading.Thread(target=periodic_metric_update, daemon=True)
    thread.start()

In [29]:

from werkzeug.serving import make_server

server = make_server('0.0.0.0', 5000, app)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print("Flask server running on http://0.0.0.0:5000")
print("Metrics available at http://0.0.0.0:5000/metrics")

Flask server running on http://0.0.0.0:5000
Metrics available at http://0.0.0.0:5000/metrics


In [30]:
# Тестовый запрос на предсказание
import requests
import json

sample = X_test[0].tolist()  # первая строка из тестовой выборки (уже масштабирована)
response = requests.post('http://localhost:5000/predict', json={'features': sample})
print(response.json())

# Проверка метрик
metrics_response = requests.get('http://localhost:5000/metrics')
print(metrics_response.text[:500])  # первые 500 символов

127.0.0.1 - - [10/Jul/2026 13:39:08] "GET /metrics HTTP/1.1" 200 -
c:\Users\Kohov\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [10/Jul/2026 13:39:09] "POST /predict HTTP/1.1" 200 -


{'prediction': 0, 'probability': [0.56, 0.44]}


127.0.0.1 - - [10/Jul/2026 13:39:11] "GET /metrics HTTP/1.1" 200 -


# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 16743.0
python_gc_objects_collected_total{generation="1"} 11203.0
python_gc_objects_collected_total{generation="2"} 36.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects


In [ ]:

indices = np.random.choice(len(X_test), size=5, replace=False)
X_new = X_test[indices]          # если X_test - numpy array, работает
y_new = y_test.iloc[indices]     # или y_test.to_numpy()[indices]

# Преобразуем в список для JSON
X_new_list = X_new.tolist()
y_new_list = y_new.tolist()

response = requests.post('http://localhost:5000/retrain', json={'X_new': X_new_list, 'y_new': y_new_list})
print(response.json())

# Проверим обновлённые метрики
metrics_response = requests.get('http://localhost:5000/metrics')
print(metrics_response.text)

c:\Users\Kohov\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [10/Jul/2026 13:39:13] "POST /retrain HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:13] "GET /metrics HTTP/1.1" 200 -


{'new_train_size': 1284, 'status': 'retrained successfully'}


127.0.0.1 - - [10/Jul/2026 13:39:15] "GET /metrics HTTP/1.1" 200 -


# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 16910.0
python_gc_objects_collected_total{generation="1"} 11203.0
python_gc_objects_collected_total{generation="2"} 36.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generation="0"} 226.0
python_gc_collections_total{generation="1"} 20.0
python_gc_collections_total{generation="2"} 1.0
# HELP python_info Python platform information
# TYPE python_info gauge
python_info{implementation="CPython",major="3",minor="13",

In [ ]:
import os
import time
import subprocess
import requests
import json


GRAFANA_URL = "http://localhost:3000"
PROMETHEUS_URL = "http://localhost:9090"      # проверим оба
PROMETHEUS_DOCKER_URL = "http://host.docker.internal:9090"
auth = ('admin', 'admin')

def check_service(url, name):
    try:
        r = requests.get(url, timeout=2)
        return True, f" {name} доступен (статус {r.status_code})"
    except Exception as e:
        return False, f" {name} НЕ доступен: {e}"

ok, msg = check_service(GRAFANA_URL, "Grafana")
print(msg)
if not ok:
    raise SystemExit("Сначала запусти Grafana: docker run -d --name=grafana -p 3000:3000 grafana/grafana")


config = """
global:
  scrape_interval: 5s

scrape_configs:
  - job_name: 'wine_model'
    static_configs:
      - targets: ['host.docker.internal:5000']
"""
with open('prometheus.yml', 'w') as f:
    f.write(config)
print("prometheus.yml создан")

# Удаляем старый контейнер
subprocess.run('docker rm -f prometheus', shell=True, capture_output=True)

# Получаем абсолютный путь 
abs_path = os.path.abspath('.').replace('\\', '/')

cmd = f'docker run -d --name=prometheus -p 9090:9090 -v "{abs_path}/prometheus.yml:/etc/prometheus/prometheus.yml" prom/prometheus'
print(f"Выполняем: {cmd}")
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("Ошибка запуска контейнера:")
    print(result.stderr)
  
    print(f' docker run -d --name=prometheus -p 9090:9090 -v "${{PWD}}/prometheus.yml:/etc/prometheus/prometheus.yml" prom/prometheus')
    raise SystemExit("Не удалось запустить Prometheus")


time.sleep(10)


ok1, msg1 = check_service(PROMETHEUS_URL, "Prometheus (localhost:9090)")
print(msg1)
ok2, msg2 = check_service(PROMETHEUS_DOCKER_URL, "Prometheus (host.docker.internal:9090)")
print(msg2)

if not ok1 and not ok2:
    print("\n Prometheus не отвечает. Смотрим логи контейнера:")
    subprocess.run('docker logs prometheus --tail 20', shell=True)

    raise SystemExit("Prometheus не запустился")

# Выбираем рабочий URL
PROMETHEUS_URL = PROMETHEUS_URL if ok1 else PROMETHEUS_DOCKER_URL
print(f"Будем использовать: {PROMETHEUS_URL}")


ds_url = f"{GRAFANA_URL}/api/datasources"
resp = requests.get(ds_url, auth=auth)
existing = [ds for ds in resp.json() if ds['name'] == 'Prometheus']
if existing:
    print("ℹ Источник данных уже есть")
else:
    payload = {
        "name": "Prometheus",
        "type": "prometheus",
        "url": PROMETHEUS_URL,
        "access": "proxy",
        "isDefault": True,
        "jsonData": {"timeInterval": "5s"}
    }
    requests.post(ds_url, json=payload, auth=auth)
    print(" Источник данных добавлен")


panels = [
    {"id": 1, "title": "Accuracy", "type": "graph", "targets": [{"expr": "model_accuracy"}], "gridPos": {"h": 8, "w": 12, "x": 0, "y": 0}},
    {"id": 2, "title": "F1 Score", "type": "graph", "targets": [{"expr": "model_f1"}], "gridPos": {"h": 8, "w": 12, "x": 12, "y": 0}},
    {"id": 3, "title": "Precision", "type": "graph", "targets": [{"expr": "model_precision"}], "gridPos": {"h": 8, "w": 12, "x": 0, "y": 8}},
    {"id": 4, "title": "Recall", "type": "graph", "targets": [{"expr": "model_recall"}], "gridPos": {"h": 8, "w": 12, "x": 12, "y": 8}},
    {"id": 5, "title": "Predictions", "type": "graph", "targets": [{"expr": "model_predictions_total"}], "gridPos": {"h": 8, "w": 24, "x": 0, "y": 16}}
]

dashboard = {
    "dashboard": {
        "title": "Model Monitoring",
        "panels": panels,
        "schemaVersion": 16,
        "version": 0
    },
    "overwrite": True
}

resp = requests.post(f"{GRAFANA_URL}/api/dashboards/db", json=dashboard, auth=auth)
if resp.status_code == 200:
    print("\nДАШБОРД СОЗДАН")
   
else:
    print("Ошибка создания дашборда:", resp.text)

 Grafana доступен (статус 200)
prometheus.yml создан
Выполняем: docker run -d --name=prometheus -p 9090:9090 -v "c:/Users/Kohov/OneDrive/Desktop/IDE/Ds_prod_proj/prometheus.yml:/etc/prometheus/prometheus.yml" prom/prometheus


127.0.0.1 - - [10/Jul/2026 13:39:24] "GET /metrics HTTP/1.1" 200 -


 Prometheus (localhost:9090) доступен (статус 200)
 Prometheus (host.docker.internal:9090) доступен (статус 200)
Будем использовать: http://localhost:9090
ℹ Источник данных уже есть

ДАШБОРД СОЗДАН


127.0.0.1 - - [10/Jul/2026 13:39:29] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:34] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:39] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:44] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:49] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:54] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:39:59] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:04] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:09] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:14] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:19] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:24] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:29] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:34] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [10/Jul/2026 13:40:39] "GET /metrics HTTP/1.1" 2